<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="500" height="250" /> 
    
    
# <font color='navy'> Project: Greeks Case

<font color='black'>

- Luis Fernando Márquez Bañuelos
- Luis Eduardo Jiménez del Muro
- Fernando López Coronado
- Diego Lozoya Morales

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt

from scipy.optimize import minimize

## <font color='cornflowerblue'> Black-Scholes Class

In [2]:
class BlackScholes:
    def __init__(self, St = None, K: float = None, T: float = None, t: float = None, r: float = None, sigma: float = None):
        """
        Initialize Black-Scholes model parameters.

        Parameters:
            St (float): Spot price
            K (float): Strike price
            T (float): Time to maturity (years)
            t (float): Current time
            r (float): Risk-free interest rate
            sigma (float): Volatility of the underlying asset
        """
        if None in (St, K, T, t, r, sigma):
            self.St = 100
            self.K = 100
            self.T = 1
            self.t = 0
            self.r = 0.05
            self.sigma = 0.2
        else:
            self.St = St
            self.K = K
            self.T = T
            self.t = t
            self.r = r
            self.sigma = sigma

    # Black-Scholes formula components
    def d1(self):
        """Compute d1 in the Black-Scholes formula."""
        return (np.log(self.St / self.K) + (self.r + 0.5 * self.sigma ** 2) * (self.T - self.t)) / (self.sigma * np.sqrt(self.T - self.t))

    def d2(self):
        """Compute d2 in the Black-Scholes formula."""
        return (np.log(self.St / self.K) + (self.r - 0.5 * self.sigma ** 2) * (self.T - self.t)) / (self.sigma * np.sqrt(self.T - self.t))

    # Cumulative distribution function for a standard normal variable
    @staticmethod
    def N(x):
        """Cumulative distribution function for a standard normal variable."""
        return st.norm.cdf(x)
    
    # Option pricing methods
    def binary_cash_or_nothing_put(self):
        """Price of a Binary Cash-or-Nothing Put option."""
        return self.K * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2())

    def binary_cash_or_nothing_call(self):
        """Price of a Binary Cash-or-Nothing Call option."""
        return self.K * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2())
    
    def binary_asset_or_nothing_put(self):
        """Price of a Binary Asset-or-Nothing Put option."""
        return self.St * self.N(-self.d1())

    def binary_asset_or_nothing_call(self):
        """Price of a Binary Asset-or-Nothing Call option."""
        return self.St * self.N(self.d1())
    
    # Financial options pricing methods (European options)
    def financial_call(self):
        """Price of a European Call option."""
        return self.St * self.N(self.d1()) - self.K * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2())
    
    def financial_put(self):
        """Price of a European Put option."""
        return self.K * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2()) - self.St * self.N(-self.d1())
    
    # Greeks options pricing methods
    def delta_call(self):
        """Delta of a European Call option."""
        return self.N(self.d1())
    
    def delta_put(self):
        """Delta of a European Put option."""
        return self.N(self.d1()) - 1
    
    def gamma(self):
        """Gamma of the option."""
        return st.norm.pdf(self.d1()) / (self.St * self.sigma * np.sqrt(self.T - self.t))
    
    def vega(self):
        """Vega of the option."""
        return self.St * st.norm.pdf(self.d1()) * np.sqrt(self.T - self.t)
    
    def theta_call(self):
        """Theta of a European Call option."""
        return (-self.St * st.norm.pdf(self.d1()) * self.sigma / (2 * np.sqrt(self.T - self.t)) - self.r * self.K * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2()))
    
    def theta_put(self):
        """Theta of a European Put option."""
        return (-self.St * st.norm.pdf(self.d1()) * self.sigma / (2 * np.sqrt(self.T - self.t)) + self.r * self.K * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2()))
    
    def rho_call(self):
        """Rho of a European Call option."""
        return self.K * (self.T - self.t) * np.exp(-self.r * (self.T - self.t)) * self.N(self.d2())
    
    def rho_put(self):
        """Rho of a European Put option."""
        return -self.K * (self.T - self.t) * np.exp(-self.r * (self.T - self.t)) * self.N(-self.d2())

    @staticmethod
    def calculate_greeks_df(df, option_type='call'):
        """
        Calculate the 5 Greeks for each row in a DataFrame.
        
        Parameters:
            df (pd.DataFrame): DataFrame with columns 'St', 'K', 'T', 't', 'r', 'sigma'
            option_type (str): Either 'call' or 'put'
        
        Returns:
            pd.DataFrame: Original DataFrame with added columns for Delta, Gamma, Vega, Theta, and Rho
        """
        if option_type.lower() not in ['call', 'put']:
            raise ValueError("option_type must be either 'call' or 'put'")
        
        # Create a copy to avoid modifying the original
        result_df = df.copy()
        
        # Initialize lists to store Greeks
        deltas = []
        gammas = []
        vegas = []
        thetas = []
        rhos = []
        
        # Calculate Greeks for each row
        for idx, row in df.iterrows():
            bs = BlackScholes(
                St=row['St'],
                K=row['K'],
                T=row['T'],
                t=row['t'],
                r=row['r'],
                sigma=row['sigma']
            )
            
            if option_type.lower() == 'call':
                deltas.append(bs.delta_call())
                thetas.append(bs.theta_call())
                rhos.append(bs.rho_call())
            else:  # put
                deltas.append(bs.delta_put())
                thetas.append(bs.theta_put())
                rhos.append(bs.rho_put())
            
            # Gamma and Vega are the same for both calls and puts
            gammas.append(bs.gamma())
            vegas.append(bs.vega())
        
        # Add Greeks as new columns
        result_df['Delta'] = deltas
        result_df['Gamma'] = gammas
        result_df['Vega'] = vegas
        result_df['Theta'] = thetas
        result_df['Rho'] = rhos
        
        return result_df

## <font color='cornflowerblue'> Data Preprocessing

In [3]:
data = pd.read_excel('options.xlsx', sheet_name=1)
data

,Contract Name,Last Trade Date (EDT),Strike,Last Price,Bid,Ask,Change,% Change,Volume,Open Interest,Implied Volatility,Type,S0,Date,Maturity,T,Implied Vol
0,AAPL251121C00100000,8/22/2025 11:56 AM,100.0,128.57,128.65,132.30,0.00,0.000,1,8,0.9421,Call,229.72,2025-09-02,2025-11-21,0.219178,0.268495
1,AAPL251121C00105000,2025-11-08 11:05:00,105.0,122.55,123.70,127.35,0.00,0.000,2,0,0.9021,Call,229.72,2025-09-02,2025-11-21,0.219178,0.264088
2,AAPL251121C00110000,7/31/2025 10:57 AM,110.0,100.22,121.20,124.90,0.00,0.000,10,1,1.1539,Call,229.72,2025-09-02,2025-11-21,0.219178,0.047704
3,AAPL251121C00115000,8/19/2025 10:12 AM,115.0,119.07,113.80,117.45,0.00,0.000,3,10,0.8250,Call,229.72,2025-09-02,2025-11-21,0.219178,1.107892
4,AAPL251121C00120000,2025-05-08 12:53:00,120.0,85.11,108.90,112.50,0.00,0.000,2,3,0.7918,Call,229.72,2025-09-02,2025-11-21,0.219178,0.051409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
786,AAPL251121C00330000,10/30/2025 3:01 PM,330.0,0.09,0.07,0.09,0.01,0.125,59,651,0.3447,Call,271.40,2025-11-20,2025-11-21,0.002740,0.341069
787,AAPL251121C00335000,10/30/2025 2:38 PM,335.0,0.05,0.00,0.07,-0.03,-0.375,102,699,0.3565,Call,271.40,2025-11-20,2025-11-21,0.002740,0.338931
788,AAPL251121C00340000,10/30/2025 1:44 PM,340.0,0.05,0.00,0.18,0.00,0.000,149,565,0.4273,Call,271.40,2025-11-20,2025-11-21,0.002740,0.359936
789,AAPL251121C00345000,10/30/2025 3:46 PM,345.0,0.04,0.04,0.18,-0.01,-0.200,96,833,0.4502,Call,271.40,2025-11-20,2025-11-21,0.002740,0.371302


In [4]:
# Drop unnecessary columns
data.drop(columns=['Contract Name','Last Trade Date (EDT)','Bid','Ask','Change','% Change','Volume','Open Interest','Implied Volatility','Type'], inplace=True)

# Merge risk-free rate data
data['r'] = 0.0382
data['t'] = 0

# Rename columns for consistency
data = data.rename(columns={'Strike': 'K', 'S0': 'St', 'Implied Vol': 'sigma', 'Rf': 'r'})

# Correct r to continuous compounding
data['r'] = np.log(1 + data['r'])

In [5]:
# Calculating proper implied volatilities with the adjusted risk-free rate
all_S0 = data['St'].values
K = data['K'].values
T = data['T'].values
t = data['t'].values
r = data['r'].values
sigma = data['sigma'].values
CallMKT = data['Last Price'].values

results = []
for i in range(len(data)):
    bounds = [[None, None]]
    apply_constraints1 = lambda x: BlackScholes(all_S0[i], K[i], T[i], t[i], r[i], x[0]).financial_call() - CallMKT[i]
    my_constraints = {'type': 'eq', 'fun': apply_constraints1}
    result = minimize(lambda x: 0.0,
                x0=np.array([sigma[i]]),
                bounds=bounds,
                constraints=my_constraints,
                method='SLSQP')
    implied_volatility = result.x[0]
    results.append(implied_volatility)

data['sigma'] = results

In [6]:
# Calculating greeks
bs = BlackScholes()
data = bs.calculate_greeks_df(data, option_type='call')
data

,K,Last Price,St,Date,Maturity,T,sigma,r,t,Delta,Gamma,Vega,Theta,Rho
0,100.0,128.57,229.72,2025-09-02,2025-11-21,0.219178,0.268495,0.037488,0,1.000000e+00,1.828770e-12,5.679243e-09,-3.718168e+00,2.173845e+01
1,105.0,122.55,229.72,2025-09-02,2025-11-21,0.219178,0.264088,0.037488,0,1.000000e+00,1.213765e-11,3.707475e-08,-3.904076e+00,2.282537e+01
2,110.0,100.22,229.72,2025-09-02,2025-11-21,0.219178,0.047704,0.037488,0,1.000000e+00,2.265810e-243,1.250182e-240,-4.089985e+00,2.391229e+01
3,115.0,119.07,229.72,2025-09-02,2025-11-21,0.219178,1.112481,0.037488,0,9.457218e-01,9.201143e-04,1.183938e+01,-3.372722e+01,2.151916e+01
4,120.0,85.11,229.72,2025-09-02,2025-11-21,0.219178,0.051409,0.037488,0,1.000000e+00,4.115869e-164,2.447341e-161,-4.461801e+00,2.608614e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
786,330.0,0.09,271.40,2025-11-20,2025-11-21,0.002740,0.341069,0.037488,0,3.869355e-28,8.805562e-28,6.060748e-26,-3.776449e-24,2.872490e-28
787,335.0,0.05,271.40,2025-11-20,2025-11-21,0.002740,0.338931,0.037488,0,1.038814e-32,2.575296e-32,1.761430e-30,-1.090586e-28,7.712835e-33
788,340.0,0.05,271.40,2025-11-20,2025-11-21,0.002740,0.359936,0.037488,0,3.384187e-33,7.961584e-33,5.782977e-31,-3.802177e-29,2.512443e-33
789,345.0,0.04,271.40,2025-11-20,2025-11-21,0.002740,0.371302,0.037488,0,3.068052e-35,7.219301e-35,5.409405e-33,-3.668673e-31,2.277742e-35


## <font color='cornflowerblue'> Gamma Threshold

In [7]:
np.random.seed(42)

# Filter strikes between 200 and 300 (reasonable range)
data = data[(data['K'] >= 200) & (data['K'] <= 300)]
# Keep only integer strikes
data = data[data['K'] % 1 == 0].reset_index(drop=True)

# Dates and Strikes
dates = data['Date'].unique()
strikes = data['K'].unique()
n_dates = len(dates)

# Create index mappings
date_to_idx = {d: i for i, d in enumerate(dates)}
strike_to_idx = {s: i for i, s in enumerate(strikes)}

# Assign indices to dates and strikes with the map created and add those to the dataframe
data['date_idx'] = data['Date'].map(date_to_idx)
data['strike_idx'] = data['K'].map(strike_to_idx)

gammas = []
for _ in range(1000):
    # Pick random strikes
    selected_strike_indices = np.random.choice(len(strikes), size=10, replace=False)
    
    # Filter data
    filtered = data[data['strike_idx'].isin(selected_strike_indices)]
    
    # Vectorized gamma sum per date
    total_gamma = filtered.groupby('date_idx')['Gamma'].sum().reindex(range(n_dates), fill_value=0).values
    
    gammas.append(total_gamma)

gammas = np.array(gammas)

In [8]:
all_gammas = gammas.flatten()

upper_threshold = np.percentile(all_gammas, 75)
lower_threshold = np.percentile(all_gammas, 25)

lower_threshold, upper_threshold

(0.08217612582697384, 0.1022501127623168)

In [9]:
thresholds = np.percentile(gammas, [25, 75], axis=0)
lower_threshold_per_date = thresholds[0]
upper_threshold_per_date = thresholds[1]